# Lesson 2c Demo: Stateless and Stateful Agents

The central challenge of conversational AI: **the API is stateless, but users expect memory.**

1. **Stateless call** — each API call is independent; the model literally forgets
2. **Stateful call** — we fix this by replaying the full message history every turn
3. **Token growth** — this fix has quadratic cost
4. **Sliding window + summarization** — bound the cost while preserving information
5. **Long-term memory** — persist facts across sessions

Run cells in order.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"
print("Client ready")


## 3) Stateless: each turn is independent

The Chat Completions API is **stateless by design** — every call is a fresh start. The model has no server-side session or hidden memory; it only sees the `messages` array you send in that request. Anything you don't include simply doesn't exist for the model.

Below we make two separate calls. Turn 1 tells the model a name; turn 2 asks for it — but since turn 2 doesn't include turn 1's messages, the model has no idea.

In [ ]:
turn1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Alice."}]
)
print("Turn 1:", turn1.choices[0].message.content)

turn2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is my name?"}]
)
print("Turn 2:", turn2.choices[0].message.content)


## 4) Stateful: send history every turn

The fix is simple: **maintain a `messages` list and append every exchange to it.** On each new turn, send the entire list to the API. The model sees the full conversation and can answer follow-up questions.

This is exactly how ChatGPT works under the hood — the client sends the growing message history with every request. The "memory" lives in your code, not on OpenAI's servers.

In [ ]:
history = []

history.append({"role": "user", "content": "My name is Alice."})
r1 = client.chat.completions.create(model=MODEL, messages=history)
a1 = r1.choices[0].message.content
history.append({"role": "assistant", "content": a1})

history.append({"role": "user", "content": "What is my name?"})
r2 = client.chat.completions.create(model=MODEL, messages=history)
a2 = r2.choices[0].message.content

print("Assistant turn 1:", a1)
print("Assistant turn 2:", a2)
print("Prompt tokens on turn 2:", r2.usage.prompt_tokens)


## 5) Measure token growth

Sending full history works, but **the cost is quadratic**. Each turn re-sends every prior turn, so input tokens grow roughly as N × (average turn length). For a conversation averaging ~150 tokens per turn, the total input tokens over N turns is approximately **150 × N²/2**.

An 8-turn conversation doesn't feel expensive, but by turn 50 you're sending ~190k input tokens per call. This is the motivation for everything that follows: we need strategies to keep the context bounded while preserving as much information as possible.

In [ ]:
prompts = [
    "My name is Alice and I live in Trento.",
    "I work in computer science.",
    "I prefer concise answers.",
    "I am teaching an AI systems class.",
    "Summarize what you know about me so far.",
    "What details might be missing?",
    "Now give me three study tips.",
    "What is my name and where do I live?",
]

history = []
stats = []

for i, p in enumerate(prompts, start=1):
    history.append({"role": "user", "content": p})
    r = client.chat.completions.create(model=MODEL, messages=history, temperature=0.2)
    reply = r.choices[0].message.content
    history.append({"role": "assistant", "content": reply})

    stats.append((i, r.usage.prompt_tokens, r.usage.completion_tokens))

print("turn | prompt_tokens | completion_tokens")
for row in stats:
    print(f"{row[0]:>4} | {row[1]:>13} | {row[2]:>17}")


## 6) Sliding window + memory summarization

The solution is a two-part strategy from `3_agent_with_memory.py`:

1. **Sliding window** — keep the last N turns verbatim so the model has precise recent context
2. **Summarizer** — compress the older (dropped) turns into a short memory string using an LLM call

What gets sent to the API on each turn: `[system message with memory summary] + [recent window] + [new user message]`. The total token count stays roughly bounded regardless of conversation length.

The tradeoff: bounded cost at the price of **lossy compression**. The summary captures the gist but loses exact wording and small details. This is acceptable for most applications — humans do the same thing.

In [ ]:
from importlib.util import spec_from_file_location, module_from_spec
from pathlib import Path

# Import helpers from 3_agent_with_memory.py
mod_path = Path('labs/02_standalone_agents/3_agent_with_memory.py')
if not mod_path.exists():
    mod_path = Path('../labs/02_standalone_agents/3_agent_with_memory.py')

spec = spec_from_file_location('l2_memory', mod_path)
mod = module_from_spec(spec)
spec.loader.exec_module(mod)

# --- A realistic 10-turn conversation (hardcoded) ---
conversation = [
    {"role": "user", "content": "Hi! I'm studying computer science at UniTN."},
    {"role": "assistant", "content": "Welcome! That's a great program. How can I help?"},
    {"role": "user", "content": "I'm working on a project about recommendation systems."},
    {"role": "assistant", "content": "Nice topic! Are you focusing on collaborative filtering or content-based?"},
    {"role": "user", "content": "Collaborative filtering, specifically matrix factorization."},
    {"role": "assistant", "content": "Classic approach. SVD-based methods are a good starting point."},
    {"role": "user", "content": "Yes, I'm implementing SVD with PyTorch."},
    {"role": "assistant", "content": "Good choice for GPU acceleration. Are you using the MovieLens dataset?"},
    {"role": "user", "content": "Exactly, MovieLens 1M. Getting decent RMSE so far."},
    {"role": "assistant", "content": "That's a solid benchmark. What RMSE are you targeting?"},
]

# Step 1: Sliding window — keep only the last 2 turns
WINDOW = 2
window = mod.trim_to_window(conversation, window_turns=WINDOW)
older = conversation[:len(conversation) - len(window)]

print(f"Full conversation: {len(conversation)} messages ({len(conversation)//2} turns)")
print(f"Window keeps last {WINDOW} turns: {len(window)} messages")
print(f"Dropped (older): {len(older)} messages\n")

# Step 2: Summarize the dropped turns (this makes an API call)
print("--- Summarizing older turns (API call) ---")
memory = mod.summarize_turns(older, existing_memory="")
print(f"Memory summary:\n{memory}\n")

# Step 3: Build the final message array
user_question = "Can you remind me what dataset I'm using and suggest a next step?"
messages = mod.build_messages(memory, window, user_question)
print(f"--- Final message array sent to API: {len(messages)} messages ---")
for m in messages:
    role = m["role"].upper()
    preview = m["content"][:120].replace("\n", " ")
    print(f"  [{role}] {preview}{'...' if len(m['content']) > 120 else ''}")

# Step 4: Make the actual API call with windowed + summarized context
print("\n--- Model response ---")
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.3)
print(response.choices[0].message.content)
print(f"\nPrompt tokens: {response.usage.prompt_tokens} (bounded, not growing with full history)")

## 7) Long-term memory: persist across sessions

Everything above is **short-term memory** — it exists only while the conversation is running. Close the window and it's gone.

**Long-term memory** persists across sessions by saving facts to disk (or a database). The key insight: we ask the LLM itself to decide what's worth remembering. Given a conversation transcript, it extracts structured facts (name, location, preferences, projects) and we write them to a JSON file keyed by user ID.

On the next session, we load the user's facts and inject them into the system prompt — the model "remembers" things from previous conversations it never actually saw.

This is the pattern behind ChatGPT's memory feature, and it requires **user identity** (you need to know *whose* facts to load).

In [ ]:
import tempfile

# Import helpers from 4_agent_with_long_term_memory.py
mod4_path = Path('labs/02_standalone_agents/4_agent_with_long_term_memory.py')
if not mod4_path.exists():
    mod4_path = Path('../labs/02_standalone_agents/4_agent_with_long_term_memory.py')

spec4 = spec_from_file_location('l2_long_term', mod4_path)
mod4 = module_from_spec(spec4)
spec4.loader.exec_module(mod4)

# Override MEMORY_FILE to a temp path (avoid leaving artifacts)
tmp_memory = Path(tempfile.mkdtemp()) / "demo_memories.json"
mod4.MEMORY_FILE = tmp_memory

# --- Simulate a conversation where the user shares personal facts ---
demo_conversation = [
    {"role": "user", "content": "Hi, my name is Marco and I'm a PhD student in Trento."},
    {"role": "assistant", "content": "Nice to meet you, Marco! What are you working on?"},
    {"role": "user", "content": "I'm researching graph neural networks for drug discovery."},
    {"role": "assistant", "content": "Fascinating area! Are you using any specific GNN frameworks?"},
    {"role": "user", "content": "Mostly PyTorch Geometric. I also prefer concise explanations over long ones."},
    {"role": "assistant", "content": "Noted! I'll keep things brief. How's the research going?"},
]

# Step 1: Extract long-term facts (API call — the LLM decides what to persist)
print("--- Extracting long-term facts (API call) ---")
existing_memory = {"facts": [], "preferences": []}
extracted = mod4.extract_long_term_facts(demo_conversation, existing_memory)
print(f"Facts:       {extracted.get('facts', [])}")
print(f"Preferences: {extracted.get('preferences', [])}")

# Step 2: Save to disk, then load back (round-trip)
user_id = "marco"
mod4.save_user_memory(user_id, extracted)
loaded = mod4.get_user_memory(user_id)
print(f"\nSaved and reloaded for '{user_id}': {loaded}")

# Clean up temp file
tmp_memory.unlink(missing_ok=True)
tmp_memory.parent.rmdir()
print(f"\n(Temp file cleaned up — no artifacts left on disk)")

## Optional: run full CLI scripts

For the full interactive experience (recommended live in the terminal):
- `uv run python labs/02_standalone_agents/1_stateless_agent.py`
- `uv run python labs/02_standalone_agents/2_stateful_agent.py`
- `uv run python labs/02_standalone_agents/3_agent_with_memory.py`
- `uv run python labs/02_standalone_agents/4_agent_with_long_term_memory.py`